# 01. Data Preparation and Quality Assessment

This notebook validates the raw DataCo supply-chain dataset, prepares a reproducible order-item-level analytical table, engineers logistics and financial features, exports quality reports, and loads the same processed table into SQLite.

**Dataset grain:** one row represents one order item. No source row is removed or imputed without a documented business rule.

In [1]:
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## 1. Resolve project paths and load the raw dataset

In [2]:
def find_project_root(start: Path) -> Path:
    """Return the nearest parent that contains the project data directory."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() or (candidate / "working_data").exists():
            return candidate
    return start

project_root = find_project_root(Path.cwd())

candidate_files = [
    project_root / "data" / "raw" / "DataCoSupplyChainDataset.csv",
    project_root / "working_data" / "DataCoSupplyChainDataset.csv",
]
raw_file = next((path for path in candidate_files if path.exists()), None)

if raw_file is None:
    searched = "\n".join(str(path) for path in candidate_files)
    raise FileNotFoundError(f"DataCoSupplyChainDataset.csv was not found. Searched:\n{searched}")

raw_df = pd.read_csv(raw_file, encoding="iso-8859-1", low_memory=False)

print(f"Source: {raw_file}")
print(f"Rows: {len(raw_df):,}")
print(f"Columns: {raw_df.shape[1]}")

Source: C:\Users\Nothing\Downloads\newgithubprojects\dataco-supplychain-analytics\data\raw\DataCoSupplyChainDataset.csv
Rows: 180,519
Columns: 53


## 2. Schema and dataset-grain validation

Required analytical columns are checked before processing. `Order Item Id` must identify the row grain, while `Order Id` is a higher-level business entity.

In [3]:
required_columns = [
    "Order Item Id", "Order Id", "Customer Id",
    "order date (DateOrders)", "shipping date (DateOrders)",
    "Order Status", "Delivery Status", "Shipping Mode",
    "Days for shipping (real)", "Days for shipment (scheduled)",
    "Late_delivery_risk", "Sales", "Order Item Total",
    "Order Profit Per Order", "Market", "Order Region",
    "Department Name", "Category Name", "Product Name",
]

missing_required_columns = sorted(set(required_columns) - set(raw_df.columns))
if missing_required_columns:
    raise KeyError(f"Missing required columns: {missing_required_columns}")

grain_summary = pd.DataFrame({
    "metric": [
        "total_rows", "unique_order_items", "unique_orders",
        "duplicate_order_item_ids", "duplicate_full_rows",
    ],
    "value": [
        len(raw_df), raw_df["Order Item Id"].nunique(),
        raw_df["Order Id"].nunique(),
        raw_df["Order Item Id"].duplicated().sum(), raw_df.duplicated().sum(),
    ],
})
display(grain_summary)

,metric,value
0,total_rows,180519
1,unique_order_items,180519
2,unique_orders,65752
3,duplicate_order_item_ids,0
4,duplicate_full_rows,0


## 3. Missing-value profile

In [4]:
missing_summary = pd.DataFrame({
    "column": raw_df.columns,
    "missing_values": raw_df.isna().sum().values,
    "missing_rate_pct": (raw_df.isna().mean().values * 100).round(4),
})
missing_summary = missing_summary.sort_values(
    ["missing_values", "column"], ascending=[False, True]
).reset_index(drop=True)

critical_columns = required_columns[:-1]
overall_completeness = raw_df.notna().mean().mean() * 100
critical_completeness = raw_df[critical_columns].notna().mean().mean() * 100

display(missing_summary.query("missing_values > 0"))
print(f"Overall completeness: {overall_completeness:.2f}%")
print(f"Critical-field completeness: {critical_completeness:.2f}%")

,column,missing_values,missing_rate_pct
0,Product Description,180519,100.00
1,Order Zipcode,155679,86.24
2,Customer Lname,8,0.00
3,Customer Zipcode,3,0.00


Overall completeness: 96.49%
Critical-field completeness: 100.00%


## 4. Data preparation and feature engineering

The raw DataFrame is preserved. All transformations are applied to `processed_df`. Canceled items are never classified as delayed or not delayed.

In [5]:
processed_df = raw_df.copy()

date_columns = ["order date (DateOrders)", "shipping date (DateOrders)"]
for column in date_columns:
    processed_df[column] = pd.to_datetime(processed_df[column], errors="coerce")

processed_df["order_date"] = processed_df["order date (DateOrders)"].dt.normalize()
processed_df["shipping_date"] = processed_df["shipping date (DateOrders)"].dt.normalize()
processed_df["order_year"] = processed_df["order_date"].dt.year.astype("Int64")
processed_df["order_month"] = processed_df["order_date"].dt.month.astype("Int64")
processed_df["order_year_month"] = processed_df["order_date"].dt.to_period("M").astype("string")

processed_df["schedule_variance_days"] = (
    processed_df["Days for shipping (real)"]
    - processed_df["Days for shipment (scheduled)"]
)

is_canceled = processed_df["Delivery Status"].eq("Shipping canceled")
is_delayed = (~is_canceled) & processed_df["schedule_variance_days"].gt(0)
is_not_delayed = (~is_canceled) & processed_df["schedule_variance_days"].le(0)

processed_df["is_canceled"] = is_canceled.astype("int8")
processed_df["is_delayed"] = is_delayed.astype("int8")
processed_df["is_not_delayed"] = is_not_delayed.astype("int8")
processed_df["delivery_timing"] = np.select(
    [is_canceled, is_delayed, is_not_delayed],
    ["Canceled", "Delayed", "Not delayed"],
    default="Unknown",
)
processed_df["loss_amount"] = (
    -processed_df["Order Profit Per Order"].clip(upper=0)
)

print(f"Processed rows: {len(processed_df):,}")
print(f"Processed columns: {processed_df.shape[1]}")
display(processed_df[[
    "Order Item Id", "order_date", "shipping_date",
    "schedule_variance_days", "delivery_timing",
    "is_canceled", "is_delayed", "is_not_delayed", "loss_amount",
]].head())

Processed rows: 180,519
Processed columns: 64


,Order Item Id,order_date,shipping_date,schedule_variance_days,delivery_timing,is_canceled,is_delayed,is_not_delayed,loss_amount
0,180517,2018-01-31,2018-02-03,-1,Not delayed,0,0,1,-0.00
1,179254,2018-01-13,2018-01-18,1,Delayed,0,1,0,249.09
2,179253,2018-01-13,2018-01-17,0,Not delayed,0,0,1,247.78
3,179252,2018-01-13,2018-01-16,-1,Not delayed,0,0,1,-0.00
4,179251,2018-01-13,2018-01-15,-2,Not delayed,0,0,1,-0.00


## 5. Business-rule quality checks

In [6]:
actual_days = processed_df["Days for shipping (real)"]
scheduled_days = processed_df["Days for shipment (scheduled)"]
not_canceled = processed_df["is_canceled"].eq(0)
expected_late_risk = processed_df["is_delayed"]

check_counts = {
    "missing_critical_values": processed_df[critical_columns].isna().any(axis=1).sum(),
    "duplicate_order_item_ids": processed_df["Order Item Id"].duplicated().sum(),
    "duplicate_full_rows": raw_df.duplicated().sum(),
    "invalid_late_risk_values": (~processed_df["Late_delivery_risk"].isin([0, 1])).sum(),
    "negative_shipping_days": ((actual_days < 0) | (scheduled_days < 0)).sum(),
    "negative_sales": (processed_df["Sales"] < 0).sum(),
    "invalid_dates": processed_df[date_columns].isna().any(axis=1).sum(),
    "shipping_before_order_date": (processed_df["shipping_date"] < processed_df["order_date"]).sum(),
    "late_risk_inconsistency": (not_canceled & processed_df["Late_delivery_risk"].ne(expected_late_risk)).sum(),
    "unknown_delivery_timing": processed_df["delivery_timing"].eq("Unknown").sum(),
}

quality_report = pd.DataFrame(
    [{"check": check, "failed_records": int(count)} for check, count in check_counts.items()]
)
quality_report["status"] = np.where(quality_report["failed_records"].eq(0), "PASS", "REVIEW")
display(quality_report)

,check,failed_records,status
0,missing_critical_values,0,PASS
1,duplicate_order_item_ids,0,PASS
2,duplicate_full_rows,0,PASS
3,invalid_late_risk_values,0,PASS
4,negative_shipping_days,0,PASS
5,negative_sales,0,PASS
6,invalid_dates,0,PASS
7,shipping_before_order_date,0,PASS
8,late_risk_inconsistency,0,PASS
9,unknown_delivery_timing,0,PASS


## 6. Data-quality dimension scores

In [7]:
row_count = len(processed_df)
uniqueness_score = 100 * (1 - (check_counts["duplicate_order_item_ids"] + check_counts["duplicate_full_rows"]) / (row_count * 2))
validity_failures = sum(check_counts[key] for key in [
    "invalid_late_risk_values", "negative_shipping_days", "negative_sales", "invalid_dates"
])
validity_score = 100 * (1 - validity_failures / (row_count * 4))
consistency_score = 100 * (1 - check_counts["late_risk_inconsistency"] / max(int(not_canceled.sum()), 1))
anomaly_score = 100 * (1 - check_counts["shipping_before_order_date"] / row_count)

dimension_scores = pd.DataFrame({
    "dimension": ["Completeness", "Uniqueness", "Validity", "Consistency", "Anomalies"],
    "score_pct": [overall_completeness, uniqueness_score, validity_score, consistency_score, anomaly_score],
}).round(2)
overall_quality_score = dimension_scores["score_pct"].mean()

display(dimension_scores)
print(f"Overall data-quality score: {overall_quality_score:.2f}%")

,dimension,score_pct
0,Completeness,96.49
1,Uniqueness,100.00
2,Validity,100.00
3,Consistency,100.00
4,Anomalies,100.00


Overall data-quality score: 99.30%


## 7. Analytical KPI validation

These metrics provide one shared reconciliation point for Python, SQL, and Power BI. Canceled items are excluded from delay-rate denominators.

In [8]:
total_items = len(processed_df)
total_orders = processed_df["Order Id"].nunique()
non_canceled_items = int((processed_df["is_canceled"] == 0).sum())
canceled_items = int(processed_df["is_canceled"].sum())
delayed_items = int(processed_df["is_delayed"].sum())
not_delayed_items = int(processed_df["is_not_delayed"].sum())
total_sales = processed_df["Sales"].sum()
total_profit = processed_df["Order Profit Per Order"].sum()

kpi_validation = pd.DataFrame({
    "metric": [
        "Total Order Items", "Total Orders", "Total Sales", "Total Profit",
        "Non-Canceled Items", "Delayed Items", "Not Delayed Items",
        "Canceled Items", "Delay Rate (%)", "Not Delayed Rate (%)",
        "Cancellation Rate (%)", "Loss-Making Items", "Loss Amount",
    ],
    "value": [
        total_items, total_orders, total_sales, total_profit, non_canceled_items,
        delayed_items, not_delayed_items, canceled_items,
        delayed_items / non_canceled_items * 100,
        not_delayed_items / non_canceled_items * 100,
        canceled_items / total_items * 100,
        int((processed_df["Order Profit Per Order"] < 0).sum()),
        processed_df["loss_amount"].sum(),
    ],
})
display(kpi_validation)

assert total_items == processed_df["Order Item Id"].nunique(), "Order-item grain is not unique."
assert total_items == non_canceled_items + canceled_items, "Cancellation partition is incomplete."
assert non_canceled_items == delayed_items + not_delayed_items, "Delivery-timing partition is incomplete."
assert processed_df["delivery_timing"].ne("Unknown").all(), "Unknown delivery timing found."

,metric,value
0,Total Order Items,"180,519.00"
1,Total Orders,"65,752.00"
2,Total Sales,"36,784,735.01"
3,Total Profit,"3,966,902.97"
4,Non-Canceled Items,"172,765.00"
5,Delayed Items,"98,977.00"
6,Not Delayed Items,"73,788.00"
7,Canceled Items,"7,754.00"
8,Delay Rate (%),57.29
9,Not Delayed Rate (%),42.71


## 8. Export processed data and quality reports

In [9]:
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

processed_file = processed_dir / "dataco_cleaned.csv"
missing_file = processed_dir / "missing_value_summary.csv"
quality_file = processed_dir / "data_quality_report.csv"
dimensions_file = processed_dir / "data_quality_dimension_summary.csv"

processed_df.to_csv(processed_file, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d %H:%M:%S")
missing_summary.to_csv(missing_file, index=False)
quality_report.to_csv(quality_file, index=False)
dimension_scores.to_csv(dimensions_file, index=False)

print(f"Processed dataset: {processed_file}")
print(f"Rows exported: {len(processed_df):,}")
print(f"Quality reports: {processed_dir}")

Processed dataset: C:\Users\Nothing\Downloads\newgithubprojects\dataco-supplychain-analytics\data\processed\dataco_cleaned.csv
Rows exported: 180,519
Quality reports: C:\Users\Nothing\Downloads\newgithubprojects\dataco-supplychain-analytics\data\processed


## 9. Load the identical processed table into SQLite and reconcile

In [10]:
database_path = project_root / "data" / "dataco_logistics.db"
database_path.parent.mkdir(parents=True, exist_ok=True)

with sqlite3.connect(database_path) as connection:
    processed_df.to_sql(
        "shipments", connection, if_exists="replace", index=False,
        chunksize=5000,
    )
    connection.execute('CREATE UNIQUE INDEX IF NOT EXISTS idx_shipments_order_item_id ON shipments("Order Item Id")')
    connection.execute('CREATE INDEX IF NOT EXISTS idx_shipments_order_id ON shipments("Order Id")')
    connection.execute('CREATE INDEX IF NOT EXISTS idx_shipments_order_date ON shipments(order_date)')
    sqlite_validation = pd.read_sql_query(
        '''
        SELECT
            COUNT(*) AS total_order_items,
            COUNT(DISTINCT "Order Item Id") AS unique_order_items,
            COUNT(DISTINCT "Order Id") AS total_orders,
            ROUND(SUM(Sales), 2) AS total_sales,
            ROUND(SUM("Order Profit Per Order"), 2) AS total_profit,
            SUM(is_canceled) AS canceled_items,
            SUM(is_delayed) AS delayed_items,
            SUM(is_not_delayed) AS not_delayed_items
        FROM shipments;
        ''',
        connection,
    )

display(sqlite_validation)
assert int(sqlite_validation.loc[0, "total_order_items"]) == len(processed_df)
print(f"SQLite database: {database_path}")
print("CSV and SQLite reconciliation passed.")

,total_order_items,unique_order_items,total_orders,total_sales,total_profit,canceled_items,delayed_items,not_delayed_items
0,180519,180519,65752,"36,784,735.01","3,966,902.97",7754,98977,73788


SQLite database: C:\Users\Nothing\Downloads\newgithubprojects\dataco-supplychain-analytics\data\dataco_logistics.db
CSV and SQLite reconciliation passed.


## Conclusion

- The analytical grain is one row per order item.
- The raw data remains unchanged; all engineered fields live in `processed_df`.
- Canceled items are excluded from delayed/not-delayed metrics.
- `data/processed/dataco_cleaned.csv` is the official source for Power BI.
- The same processed table is loaded into `data/dataco_logistics.db` for SQL analysis.
- Quality reports document issues instead of silently deleting or imputing records.